## **CASE RIO GALEÃO**

Objetivo: Construção de uma base mensal consolidada para voos do Galeão
(SBGL), que permita análises comparativas entre planejamento e realizado, bem como análise da qualidade dos dados, servindo como insumo estruturado para futuras avaliações estratégicas.

O produto final deve ser capaz de responder perguntas referente ao período entre JUL/2025 e JAN/2026 envolvendo vôos que tenham o aeroporto como destino ou chegada.


### **1. Importação dos pacotes**


In [541]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests
import json
import io

pd.set_option('display.max_columns', None)

### **2. Camada Bronze**
#### **2.1 Importação dos Datasets**

**SIROS - Movimentos:**  Cada linha dessa base é uma movimentação(ANAC: Termo genérico utilizado para caracterizar um pouso, uma decolagem, ou um toque e arremetida de aeronaves no aeródromo). Foi necessário extrair o relatório mês-a-mês para não atingir o limite máximo de linhas. No momento de desenvolvimento a API não estava funcionando corretamente.

In [542]:
# Listas de relatórios mensais
arquivos_destino = [
    'DESTINO_SBGL_202507_ANAC_VDF07032026035201', 'DESTINO_SBGL_202508_ANAC_VDF07032026035216',
    'DESTINO_SBGL_202509_ANAC_VDF07032026035255', 'DESTINO_SBGL_202510_ANAC_VDF07032026035346',
    'DESTINO_SBGL_202511_ANAC_VDF07032026035402', 'DESTINO_SBGL_202512_ANAC_VDF07032026035420',
    'DESTINO_SBGL_202601_ANAC_VDF07032026035452'
]

arquivos_origem = [
    'ORIGEM_SBGL_202507_ANAC_VDF07032026034804', 'ORIGEM_SBGL_202508_ANAC_VDF07032026034846',
    'ORIGEM_SBGL_202509_ANAC_VDF07032026034910', 'ORIGEM_SBGL_202510_ANAC_VDF07032026035003',
    'ORIGEM_SBGL_202511_ANAC_VDF07032026035017', 'ORIGEM_SBGL_202512_ANAC_VDF07032026035051',
    'ORIGEM_SBGL_202601_ANAC_VDF07032026035114'
]

todos_arquivos = arquivos_destino + arquivos_origem

# Importação e concatenação dos datasets mensais em um único DataFrame

lista_df = []
base_url = "https://raw.githubusercontent.com/rdgdelfino/case_riogaleao/refs/heads/main/"

for arquivo in todos_arquivos:
    url_completa = f"{base_url}{arquivo}.csv"
    df_temp = pd.read_csv(url_completa, sep=';', encoding='latin-1')
    lista_df.append(df_temp)

siros_mov_raw = pd.concat(lista_df, ignore_index=True)
siros_mov_raw.head(3)

,ICAO,Operador Aéreo,Etapa,Voo,Equip,Assentos,Origem,Aeroporto Origem,Início,Partida Prevista,Chegada Prevista,Tipo,Destino,Aeroporto Destino,Codeshare
0,CQB,APUÍ TÁXI AÉREO S/A,1,0001,E110,15,SBEG,EDUARDO GOMES - MANAUS - AM - BRASIL,01/07/2025,01/07/2025 07:55,01/07/2025 09:15,REGULAR DE PASSAGEIROS DOMÉSTICA,SWYN,PRAINHA - APUÍ - AM - BRASIL,NaN
1,CQB,APUÍ TÁXI AÉREO S/A,1,0002,E110,15,SWYN,PRAINHA - APUÍ - AM - BRASIL,01/07/2025,01/07/2025 09:50,01/07/2025 10:30,REGULAR DE PASSAGEIROS DOMÉSTICA,SBMY,MANICORÉ - MANICORÉ - AM - BRASIL,NaN
2,CQB,APUÍ TÁXI AÉREO S/A,1,0003,E110,15,SBMY,MANICORÉ - MANICORÉ - AM - BRASIL,01/07/2025,01/07/2025 11:00,01/07/2025 12:00,REGULAR DE PASSAGEIROS DOMÉSTICA,SBEG,EDUARDO GOMES - MANAUS - AM - BRASIL,NaN


**SIROS - Operação Regular:** Malha planejada e registrada pelas empresas aéreas. Cada linha é uma operação regular correspondente a um número de voo, com os dias da semana aos quais esse voo opera, constando o dia de início e fim daquela operação regular.
Nessa base constam os registros de malha da temporada atual e a futura.

In [543]:
# Registros de Operações Regulares. Importação via URL
url = 'https://raw.githubusercontent.com/rdgdelfino/case_riogaleao/refs/heads/main/siros_registros.csv'
siros_registro_raw = pd.read_csv(url, sep=';',skiprows=1)
siros_registro_raw.head(3)

,Cód. Empresa,Empresa,Nº Voo,Equip.,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Quant. Assentos,Nº SIROS,Situação SIROS,Data Registro,Início Operação,Fim Operação,Natureza Operação,Nº Etapa,Cód. Origem,Arpt Origem,Cód Destino,Arpt Destino,Horário Partida,Horário Chegada,Tipo Serviço,Objeto Transporte,Codeshare
0,AAL,"AMERICAN AIRLINES, INC.",0930,B77W,1,2,3,4,5,6,7,318,AAL-0000000000039122773,Em Operação,23/10/2025 13:10:56,2025-12-04,2026-03-28,INTERNACIONAL,1,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",02:30,11:00,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
1,AAL,"AMERICAN AIRLINES, INC.",0905,B772,1,2,3,4,5,6,7,288,AAL-0000000000039754058,Em Operação,12/11/2025 11:31:30,2026-01-07,2026-03-28,INTERNACIONAL,1,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,04:05,12:25,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
2,AAL,"AMERICAN AIRLINES, INC.",0925,B788,1,2,3,4,5,6,7,295,AAL-0000000000040317706,Em Operação,01/12/2025 15:47:47,2026-01-07,2026-03-08,INTERNACIONAL,1,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,03:30,11:50,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN


**Dados Estatísticos ANAC:** dados de voos realizados, consolidados e resumidos, de 2000 a 2026. Cada linha é um resumo da operação por empresa, origem, destino, natureza, tipo de voo, mês e ano, entre outras informações, porém sem o detalhe de número de voo, com os valores de decolagens, assentos ofertados, passageiros pagos e grátis, entre outros valores.

In [544]:
# Importação via URL
url = 'https://raw.githubusercontent.com/rdgdelfino/case_riogaleao/refs/heads/main/anac_2026.csv'
anac_2026_raw = pd.read_csv(url, sep=';', encoding='latin-1')

In [545]:
# Importação via URL
url = 'https://raw.githubusercontent.com/rdgdelfino/case_riogaleao/refs/heads/main/anac_2025.csv'
anac_2025_raw = pd.read_csv(url, sep=';', encoding='latin-1')

In [546]:
# Junção dos datasets parciais da ANAC em um único Dataset
anac_raw = pd.concat([anac_2025_raw, anac_2026_raw], ignore_index=True)
anac_raw.head(3)

,EMPRESA (SIGLA),EMPRESA (NOME),EMPRESA (NACIONALIDADE),ANO,MÊS,AEROPORTO DE ORIGEM (SIGLA),AEROPORTO DE ORIGEM (NOME),AEROPORTO DE ORIGEM (UF),AEROPORTO DE ORIGEM (REGIÃO),AEROPORTO DE ORIGEM (PAÍS),AEROPORTO DE ORIGEM (CONTINENTE),AEROPORTO DE DESTINO (SIGLA),AEROPORTO DE DESTINO (NOME),AEROPORTO DE DESTINO (UF),AEROPORTO DE DESTINO (REGIÃO),AEROPORTO DE DESTINO (PAÍS),AEROPORTO DE DESTINO (CONTINENTE),NATUREZA,GRUPO DE VOO,PASSAGEIROS PAGOS,PASSAGEIROS GRÁTIS,CARGA PAGA (KG),CARGA GRÁTIS (KG),CORREIO (KG),ASK,RPK,ATK,RTK,COMBUSTÍVEL (LITROS),DISTÂNCIA VOADA (KM),DECOLAGENS,CARGA PAGA KM,CARGA GRATIS KM,CORREIO KM,ASSENTOS,PAYLOAD,HORAS VOADAS,BAGAGEM (KG)
0,1ED,SERVICIOS AÉREOS PANAMERICANOS LTDA. SARPA S.A.S,ESTRANGEIRA,2025,4,SBEG,MANAUS,AM,NORTE,BRASIL,AMÉRICA DO SUL,SKBO,BOGOTÁ,NaN,NaN,COLÔMBIA,AMÉRICA DO SUL,INTERNACIONAL,IMPRODUTIVO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1787.0,1.0,0.0,0.0,0.0,0.0,0.0,"2,5",0.0
1,1ED,SERVICIOS AÉREOS PANAMERICANOS LTDA. SARPA S.A.S,ESTRANGEIRA,2025,4,SBEG,MANAUS,AM,NORTE,BRASIL,AMÉRICA DO SUL,SYCJ,GEORGETOWN,NaN,NaN,GUIANA,AMÉRICA DO SUL,INTERNACIONAL,IMPRODUTIVO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1079.0,1.0,0.0,0.0,0.0,0.0,0.0,"1,533",0.0
2,1ED,SERVICIOS AÉREOS PANAMERICANOS LTDA. SARPA S.A.S,ESTRANGEIRA,2025,4,SKBO,BOGOTÁ,NaN,NaN,COLÔMBIA,AMÉRICA DO SUL,SBEG,MANAUS,AM,NORTE,BRASIL,AMÉRICA DO SUL,INTERNACIONAL,NÃO REGULAR,22.0,0.0,0.0,0.0,0.0,89350.0,39314.0,8935.0,3538.0,0.0,1787.0,1.0,0.0,0.0,0.0,50.0,5000.0,"2,65",0.0


**VRA – Voo Regular Ativo:** Base que controla as operações realizadas, tendo como referência o que foi programado, onde cada linha é uma operação/movimentação/decolagem com número de voo, empresa, origem, destino, horário de operação, entre outras informações.

Obs: Devido ao tempo de processamento e memória, os dados foram extraídos, carregados e filtrados para vôos que envolvem o aeroporto do Galeão. Depois, salvo em CSV para execução e testes posteriores.

In [547]:
# Importação dos dados via API

# dt_referencia1='01072025'
# dt_referencia2='31012026'
# url = f"https://sas.anac.gov.br/sas/vra_api/vra?dt_referencia1={dt_referencia1}&dt_referencia2={dt_referencia2}"
# dados_vra_str = requests.get(url).json()
# dados_vra_json = json.loads(dados_vra_str)

# Criação do DataFrame completo e filtro dos vôos do Galeão

# vra_raw = pd.DataFrame(dados_vra_json)
# vra_galeao_raw = vra_raw[(vra_raw['sg_icao_origem'] == 'SBGL') | (vra_raw['sg_icao_destino'] == 'SBGL')]

# Download
# vra_galeao_raw.to_csv('vra_galeao_raw.csv', index=False)




In [548]:
# Importação via URL
url = 'https://raw.githubusercontent.com/rdgdelfino/case_riogaleao/refs/heads/main/vra_galeao_raw.csv'
vra_galeao_raw = pd.read_csv(url)
vra_galeao_raw.head(3)

,sg_empresa_icao,nm_empresa,nr_voo,cd_di,cd_tipo_linha,sg_equipamento_icao,nr_assentos_ofertados,sg_icao_origem,nm_aerodromo_origem,dt_partida_prevista,dt_partida_real,sg_icao_destino,nm_aerodromo_destino,dt_chegada_prevista,dt_chegada_real,ds_situacao_voo,ds_justificativa,dt_referencia,ds_situacao_partida,ds_situacao_chegada
0,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,288,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,01/07/2025 23:00,01/07/2025 22:55,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",02/07/2025 07:35,02/07/2025 07:37,REALIZADO,NaN,01/07/2025,Antecipado,Pontual
1,AAL,"AMERICAN AIRLINES, INC.",0905,0,I,B772,288,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/07/2025 00:00,01/07/2025 00:07,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,01/07/2025 08:20,01/07/2025 08:34,REALIZADO,NaN,01/07/2025,Pontual,Pontual
2,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,288,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,02/07/2025 23:00,02/07/2025 22:51,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",03/07/2025 07:35,03/07/2025 07:40,REALIZADO,NaN,02/07/2025,Antecipado,Pontual


### **3. Camada Silver**
Análise exploratória e tratamento de dados
#### 3.1 ANAC

In [549]:
anac_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41417 entries, 0 to 41416
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   EMPRESA (SIGLA)                    41417 non-null  object 
 1   EMPRESA (NOME)                     41417 non-null  object 
 2   EMPRESA (NACIONALIDADE)            41417 non-null  object 
 3   ANO                                41417 non-null  int64  
 4   MÊS                                41417 non-null  int64  
 5   AEROPORTO DE ORIGEM (SIGLA)        41417 non-null  object 
 6   AEROPORTO DE ORIGEM (NOME)         41417 non-null  object 
 7   AEROPORTO DE ORIGEM (UF)           32005 non-null  object 
 8   AEROPORTO DE ORIGEM (REGIÃO)       32005 non-null  object 
 9   AEROPORTO DE ORIGEM (PAÍS)         41417 non-null  object 
 10  AEROPORTO DE ORIGEM (CONTINENTE)   41417 non-null  object 
 11  AEROPORTO DE DESTINO (SIGLA)       41417 non-null  obj

In [550]:
# Filtro operações no Galeão
anac_silver_aux = anac_raw[((anac_raw['AEROPORTO DE ORIGEM (SIGLA)'] == 'SBGL') | (anac_raw['AEROPORTO DE DESTINO (SIGLA)'] == 'SBGL')) & 
(((anac_raw['ANO'] == 2025) & (anac_raw['MÊS'] >= 7)) | ((anac_raw['ANO'] == 2026) & (anac_raw['MÊS'] == 1)))]
print(f"O tamanho original do dataset era de {len(anac_raw)} e agora é de {len(anac_silver_aux)} operações")

O tamanho original do dataset era de 41417 e agora é de 2377 operações


In [551]:
anac_silver_aux.head(3)

,EMPRESA (SIGLA),EMPRESA (NOME),EMPRESA (NACIONALIDADE),ANO,MÊS,AEROPORTO DE ORIGEM (SIGLA),AEROPORTO DE ORIGEM (NOME),AEROPORTO DE ORIGEM (UF),AEROPORTO DE ORIGEM (REGIÃO),AEROPORTO DE ORIGEM (PAÍS),AEROPORTO DE ORIGEM (CONTINENTE),AEROPORTO DE DESTINO (SIGLA),AEROPORTO DE DESTINO (NOME),AEROPORTO DE DESTINO (UF),AEROPORTO DE DESTINO (REGIÃO),AEROPORTO DE DESTINO (PAÍS),AEROPORTO DE DESTINO (CONTINENTE),NATUREZA,GRUPO DE VOO,PASSAGEIROS PAGOS,PASSAGEIROS GRÁTIS,CARGA PAGA (KG),CARGA GRÁTIS (KG),CORREIO (KG),ASK,RPK,ATK,RTK,COMBUSTÍVEL (LITROS),DISTÂNCIA VOADA (KM),DECOLAGENS,CARGA PAGA KM,CARGA GRATIS KM,CORREIO KM,ASSENTOS,PAYLOAD,HORAS VOADAS,BAGAGEM (KG)
101,AAL,"AMERICAN AIRLINES, INC.",ESTRANGEIRA,2025,7,KDFW,"DALLAS & FORT WORTH, TEXAS",NaN,NaN,ESTADOS UNIDOS DA AMÉRICA,AMÉRICA DO NORTE,SBGL,RIO DE JANEIRO,RJ,SUDESTE,BRASIL,AMÉRICA DO SUL,INTERNACIONAL,REGULAR,0.0,0.0,0.0,0.0,0.0,2399415.0,1877437.0,378855.0,216410.0,0.0,8419.0,1.0,4.627924e+07,0.0,1161822.0,285.0,45000.0,"10,383",0.0
104,AAL,"AMERICAN AIRLINES, INC.",ESTRANGEIRA,2025,7,KMIA,"MIAMI, FLORIDA",NaN,NaN,ESTADOS UNIDOS DA AMÉRICA,AMÉRICA DO NORTE,SBGL,RIO DE JANEIRO,RJ,SUDESTE,BRASIL,AMÉRICA DO SUL,INTERNACIONAL,REGULAR,7844.0,197.0,196367.0,0.0,13827.0,56852688.0,52688148.0,9437385.0,6153794.0,0.0,208227.0,31.0,1.318997e+09,0.0,92875959.0,8464.0,1405000.0,"255,917",0.0
107,AAL,"AMERICAN AIRLINES, INC.",ESTRANGEIRA,2025,7,SBGL,RIO DE JANEIRO,RJ,SUDESTE,BRASIL,AMÉRICA DO SUL,KMIA,"MIAMI, FLORIDA",NaN,NaN,ESTADOS UNIDOS DA AMÉRICA,AMÉRICA DO NORTE,INTERNACIONAL,REGULAR,8029.0,250.0,303464.0,0.0,3787.0,55025664.0,52264977.0,9135120.0,6756905.0,0.0,201510.0,30.0,2.027634e+09,0.0,25437279.0,8192.0,1360000.0,"252,483",0.0


In [552]:
# Sumarização
anac_silver = (anac_silver_aux.groupby(by=['EMPRESA (SIGLA)','EMPRESA (NOME)','ANO','MÊS',
               'AEROPORTO DE ORIGEM (SIGLA)', 'AEROPORTO DE DESTINO (SIGLA)', 'NATUREZA']).sum().reset_index())

# Campos calculdos
anac_silver['ANAC_PASSAGEIROS'] = anac_silver['PASSAGEIROS PAGOS'] + anac_silver['PASSAGEIROS GRÁTIS'] 
anac_silver['AEROPORTO MERCADO (SIGLA)'] = np.where(anac_silver['AEROPORTO DE ORIGEM (SIGLA)'] == 'SBGL', 
    anac_silver['AEROPORTO DE DESTINO (SIGLA)'], anac_silver['AEROPORTO DE ORIGEM (SIGLA)'])

# Formatando os campos
anac_silver = anac_silver.rename(columns={'DECOLAGENS': 'ANAC_DECOLAGEM', 'ASSENTOS': 'ANAC_ASSENTOS'})

# Seleção de campos para tabela final
colunas_finais = ['EMPRESA (SIGLA)','EMPRESA (NOME)','ANO','MÊS', 'AEROPORTO DE ORIGEM (SIGLA)','AEROPORTO DE DESTINO (SIGLA)', 
                  'AEROPORTO MERCADO (SIGLA)', 'NATUREZA', 'ANAC_PASSAGEIROS', 'ANAC_DECOLAGEM', 'ANAC_ASSENTOS']

anac_silver = anac_silver[colunas_finais]
anac_silver.head(3)

,EMPRESA (SIGLA),EMPRESA (NOME),ANO,MÊS,AEROPORTO DE ORIGEM (SIGLA),AEROPORTO DE DESTINO (SIGLA),AEROPORTO MERCADO (SIGLA),NATUREZA,ANAC_PASSAGEIROS,ANAC_DECOLAGEM,ANAC_ASSENTOS
0,AAL,"AMERICAN AIRLINES, INC.",2025,7,KDFW,SBGL,KDFW,INTERNACIONAL,0.0,1.0,285.0
1,AAL,"AMERICAN AIRLINES, INC.",2025,7,KMIA,SBGL,KMIA,INTERNACIONAL,8041.0,31.0,8464.0
2,AAL,"AMERICAN AIRLINES, INC.",2025,7,SBGL,KMIA,KMIA,INTERNACIONAL,8279.0,30.0,8192.0


#### **3.2 Siros**
##### **3.2.1 Siros - Registros**
Os registros do SIROS contemplam operações de diferentes períodos e aeroportos. Necessário filtrar os vôos que envolvem o aeroporto do Galeão, dentro do período de análise (07/2025-01/2026). Este filtro reduz o número de operações regulares no dataframe de 59.357 para 42 operações regulares.

O número de operações para o período parece pouco para um aeroporto do porte do Galeão, por isso será feita uma validação entre os dados do SIROS e da ANAC.

In [553]:
siros_registro_silver = siros_registro_raw[((siros_registro_raw['Cód. Origem']	== 'SBGL') | (siros_registro_raw['Cód Destino'] == 'SBGL')) & (siros_registro_raw['Fim Operação'] >= '2025-07-01') &
    (siros_registro_raw['Início Operação'] <= '2026-01-31')]
print(f"O número de registros após o filtro diminuiu de {len(siros_registro_raw)} para {len(siros_registro_silver)}")
siros_registro_silver.head(3)

O número de registros após o filtro diminuiu de 59357 para 42


,Cód. Empresa,Empresa,Nº Voo,Equip.,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Quant. Assentos,Nº SIROS,Situação SIROS,Data Registro,Início Operação,Fim Operação,Natureza Operação,Nº Etapa,Cód. Origem,Arpt Origem,Cód Destino,Arpt Destino,Horário Partida,Horário Chegada,Tipo Serviço,Objeto Transporte,Codeshare
1,AAL,"AMERICAN AIRLINES, INC.",0905,B772,1,2,3,4,5,6,7,288,AAL-0000000000039754058,Em Operação,12/11/2025 11:31:30,2026-01-07,2026-03-28,INTERNACIONAL,1,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,04:05,12:25,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
2349,AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,0197,A20N,1,2,3,4,5,6,7,180,AVA-0000000000038909414,Em Operação,16/10/2025 12:36:22,2025-11-01,2026-03-28,INTERNACIONAL,1,SKBO,EL DORADO INTERNATIONAL AIRPORT - BOGOTÁ - COL...,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,21:05,03:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
2350,AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,0198,A20N,1,2,3,4,5,6,7,180,AVA-0000000000038909415,Em Operação,16/10/2025 12:36:22,2025-11-01,2026-03-28,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,SKBO,EL DORADO INTERNATIONAL AIRPORT - BOGOTÁ - COL...,04:45,11:05,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN


In [554]:
# Agrupamento e contagem de movimento por Origem, Destino. Ordenado de forma descrescente
siros_silver_decolagem = siros_silver.groupby(['Cód. Empresa','Empresa','Cód. Origem', 'Cód Destino']).size().reset_index(name='SIROS_DECOLAGENS').sort_values(by='SIROS_DECOLAGENS', ascending=False)
siros_silver_decolagem.head()

,Cód. Empresa,Empresa,Cód. Origem,Cód Destino,SIROS_DECOLAGENS
25,SKU,SKY AIRLINE S.A.,SCEL,SBGL,3
23,SKU,SKY AIRLINE S.A.,SBGL,SCEL,3
2,AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,SKBO,SBGL,2
1,AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,SBGL,SKBO,2
14,GTI,ATLAS AIR INC,KMIA,SBGL,2


##### Caso SIROS x ANAC

Para validar os dados entre as duas bases, a mesma operação de um vôo da Air Canada com trajeto SBGL => CYYZ de Janeiro/26 foi buscado em ambas as bases. Embora na base da ANAC existam registros para operção em Dez/2025 e Jan/2026, na base Siros as operações com as mesmas características só começarão a partir de 05/05/2026, ou seja, fora do período de apuração. Isto indica falta de integridade em uma das bases ou regras de negócio diferentes para cada uma delas. Por isso, optou-se por usar a base de movimentos e não a de registros.

In [555]:
anac_silver[(anac_silver['EMPRESA (SIGLA)']=='ACA') & (anac_silver['AEROPORTO DE ORIGEM (SIGLA)'] == 'CYYZ')]

,EMPRESA (SIGLA),EMPRESA (NOME),ANO,MÊS,AEROPORTO DE ORIGEM (SIGLA),AEROPORTO DE DESTINO (SIGLA),AEROPORTO MERCADO (SIGLA),NATUREZA,ANAC_PASSAGEIROS,ANAC_DECOLAGEM,ANAC_ASSENTOS
30,ACA,AIR CANADA,2025,12,CYYZ,SBGL,CYYZ,INTERNACIONAL,3413.0,12.0,3576.0
32,ACA,AIR CANADA,2026,1,CYYZ,SBGL,CYYZ,INTERNACIONAL,3530.0,13.0,3874.0


In [556]:
siros_registro_raw[(siros_registro_raw['Cód. Empresa']=='ACA') & (siros_registro_raw['Cód. Origem']=='SBGL')]

,Cód. Empresa,Empresa,Nº Voo,Equip.,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Quant. Assentos,Nº SIROS,Situação SIROS,Data Registro,Início Operação,Fim Operação,Natureza Operação,Nº Etapa,Cód. Origem,Arpt Origem,Cód Destino,Arpt Destino,Horário Partida,Horário Chegada,Tipo Serviço,Objeto Transporte,Codeshare
192,ACA,AIR CANADA,0085,B789,0,0,0,4,0,6,0,298,ACA-0000000000042531506,Em Operação,01/03/2026 23:35:34,2026-03-05,2026-03-07,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,00:00,10:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,"AZU/7809 início: 06/12/2025 fim: 27/03/2026, G..."
195,ACA,AIR CANADA,0085,B789,0,0,3,0,5,0,7,298,ACA-0000000000042531507,A Operar,01/03/2026 23:35:34,2026-03-08,2026-03-28,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,23:00,09:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,"GLO/6825 início: 06/12/2025 fim: 27/03/2026, A..."
231,ACA,AIR CANADA,0085,B789,0,0,3,0,5,0,0,298,ACA-0000000000042531538,A Operar,01/03/2026 23:35:57,2026-10-28,2026-10-31,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,23:00,09:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
236,ACA,AIR CANADA,0085,B789,1,0,0,4,0,6,0,298,ACA-0000000000042531539,A Operar,01/03/2026 23:35:57,2026-11-02,2027-03-13,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,00:00,10:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN
245,ACA,AIR CANADA,0085,B789,0,0,3,0,5,0,7,298,ACA-0000000000042531540,A Operar,01/03/2026 23:35:57,2027-03-14,2027-03-27,INTERNACIONAL,1,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,23:00,09:40,REGULAR DE PASSAGEIROS,PASSAGEIROS,NaN


##### **3.2.2 Siros - Movimentos**

Apesar do filtro no momento da extração dos registros, os arquivos vieram com movimentos de aeroportos que não são o Galeão. Será necessário filtrar estes registros.

In [557]:
siros_mov_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1241668 entries, 0 to 1241667
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   ICAO               1241668 non-null  object
 1   Operador Aéreo     1241668 non-null  object
 2   Etapa              1241668 non-null  int64 
 3   Voo                1241668 non-null  object
 4   Equip              1241668 non-null  object
 5   Assentos           1241668 non-null  int64 
 6   Origem             1241668 non-null  object
 7   Aeroporto Origem   1241668 non-null  object
 8   Início             1241668 non-null  object
 9   Partida Prevista   1241668 non-null  object
 10  Chegada Prevista   1241668 non-null  object
 11  Tipo               1241668 non-null  object
 12  Destino            1241668 non-null  object
 13  Aeroporto Destino  1241668 non-null  object
 14  Codeshare          526152 non-null   object
dtypes: int64(2), object(13)
memory usage: 142.1+ MB


In [558]:
siros_mov_raw.head(3)

,ICAO,Operador Aéreo,Etapa,Voo,Equip,Assentos,Origem,Aeroporto Origem,Início,Partida Prevista,Chegada Prevista,Tipo,Destino,Aeroporto Destino,Codeshare
0,CQB,APUÍ TÁXI AÉREO S/A,1,0001,E110,15,SBEG,EDUARDO GOMES - MANAUS - AM - BRASIL,01/07/2025,01/07/2025 07:55,01/07/2025 09:15,REGULAR DE PASSAGEIROS DOMÉSTICA,SWYN,PRAINHA - APUÍ - AM - BRASIL,NaN
1,CQB,APUÍ TÁXI AÉREO S/A,1,0002,E110,15,SWYN,PRAINHA - APUÍ - AM - BRASIL,01/07/2025,01/07/2025 09:50,01/07/2025 10:30,REGULAR DE PASSAGEIROS DOMÉSTICA,SBMY,MANICORÉ - MANICORÉ - AM - BRASIL,NaN
2,CQB,APUÍ TÁXI AÉREO S/A,1,0003,E110,15,SBMY,MANICORÉ - MANICORÉ - AM - BRASIL,01/07/2025,01/07/2025 11:00,01/07/2025 12:00,REGULAR DE PASSAGEIROS DOMÉSTICA,SBEG,EDUARDO GOMES - MANAUS - AM - BRASIL,NaN


In [559]:
# Filtrar apenas os registros referentes ao Galeão
siros_mov_silver_aux = siros_mov_raw.copy()
siros_mov_silver_aux = siros_mov_silver_aux[(siros_mov_silver_aux['Origem'] == 'SBGL') | (siros_mov_silver_aux['Destino'] == 'SBGL')]
print(f"O tamanho original do dataset era de {len(siros_mov_raw)} e agora é de {len(siros_mov_silver_aux)} movimentos")

# Como a base foi gerada com muitas junções, garantir a integridade e remoção de duplicadas
tamanho_full = len(siros_mov_silver_aux)
siros_mov_silver_aux =siros_mov_silver_aux.drop_duplicates()
tamanho_nodup = len(siros_mov_silver_aux)
print(f"O tamanho original do dataset era de {tamanho_full}, após a remoção de duplicidade é de {tamanho_nodup} registros.")

# Conversão de datas
siros_mov_silver_aux['Início'] = pd.to_datetime(siros_mov_silver_aux['Início'],dayfirst=True)
siros_mov_silver_aux['Partida Prevista'] = pd.to_datetime(siros_mov_silver_aux['Partida Prevista'],dayfirst=True)
siros_mov_silver_aux['Chegada Prevista'] = pd.to_datetime(siros_mov_silver_aux['Chegada Prevista'],dayfirst=True)

# Criação dos Campos ANO e MÊS
siros_mov_silver_aux['Ano'] = siros_mov_silver_aux['Partida Prevista'].dt.year
siros_mov_silver_aux['Mês'] = siros_mov_silver_aux['Partida Prevista'].dt.month

data_inicio = siros_mov_silver_aux['Partida Prevista'].min()
data_fim = siros_mov_silver_aux['Partida Prevista'].max()

# Validação dos cortes superior e inferior do intervalo.
print(f"Início: {data_inicio} | Fim: {data_fim}")

O tamanho original do dataset era de 1241668 e agora é de 145092 movimentos
O tamanho original do dataset era de 145092, após a remoção de duplicidade é de 72546 registros.
Início: 2025-07-01 00:00:00 | Fim: 2026-02-01 08:25:00


In [560]:
# Validação dos valores nulos tratamento
siros_mov_silver_aux.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72546 entries, 43 to 620830
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ICAO               72546 non-null  object        
 1   Operador Aéreo     72546 non-null  object        
 2   Etapa              72546 non-null  int64         
 3   Voo                72546 non-null  object        
 4   Equip              72546 non-null  object        
 5   Assentos           72546 non-null  int64         
 6   Origem             72546 non-null  object        
 7   Aeroporto Origem   72546 non-null  object        
 8   Início             72546 non-null  datetime64[ns]
 9   Partida Prevista   72546 non-null  datetime64[ns]
 10  Chegada Prevista   72546 non-null  datetime64[ns]
 11  Tipo               72546 non-null  object        
 12  Destino            72546 non-null  object        
 13  Aeroporto Destino  72546 non-null  object        
 14  Codeshare

In [561]:
# Filtrar campos chave e campos para agregação
siros_mov_silver_temp = siros_mov_silver_aux[['ICAO','Operador Aéreo','Ano','Mês','Origem','Destino', 'Assentos']]

# Somar o valor de assentos e contar o número de vezes que determinado movimento se repete
siros_mov_silver = (siros_mov_silver_temp.groupby(['ICAO','Operador Aéreo','Ano','Mês','Origem','Destino'])
                    .agg(SIROS_ASSENTOS=('Assentos', 'sum'), SIROS_DECOLAGEM=('ICAO', 'count')).reset_index())

siros_mov_silver.head(5)

,ICAO,Operador Aéreo,Ano,Mês,Origem,Destino,SIROS_ASSENTOS,SIROS_DECOLAGEM
0,AAL,"AMERICAN AIRLINES, INC.",2025,7,KMIA,SBGL,8928,31
1,AAL,"AMERICAN AIRLINES, INC.",2025,7,SBGL,KMIA,8928,31
2,AAL,"AMERICAN AIRLINES, INC.",2025,8,KMIA,SBGL,8928,31
3,AAL,"AMERICAN AIRLINES, INC.",2025,8,SBGL,KMIA,8928,31
4,AAL,"AMERICAN AIRLINES, INC.",2025,9,KMIA,SBGL,8640,30


##### 3.3 VRA

O dataset já havia sido filtrado anteriormente para ter apenas observações referentes ao período e ao Galeão. Faz-se o mesmo teste do vôo da Air Canada. Confirmando que de fato houve operação.

In [562]:
vra_galeao_raw[(vra_galeao_raw['sg_empresa_icao']=='ACA') & (vra_galeao_raw['sg_icao_origem']=='SBGL') & (vra_galeao_raw['sg_icao_destino']=='CYYZ')].head(3)

,sg_empresa_icao,nm_empresa,nr_voo,cd_di,cd_tipo_linha,sg_equipamento_icao,nr_assentos_ofertados,sg_icao_origem,nm_aerodromo_origem,dt_partida_prevista,dt_partida_real,sg_icao_destino,nm_aerodromo_destino,dt_chegada_prevista,dt_chegada_real,ds_situacao_voo,ds_justificativa,dt_referencia,ds_situacao_partida,ds_situacao_chegada
812,ACA,AIR CANADA,0085,0,I,B789,298,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,05/12/2025 21:00,05/12/2025 21:26,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,06/12/2025 07:40,06/12/2025 07:58,REALIZADO,NaN,05/12/2025,Pontual,Pontual
814,ACA,AIR CANADA,0085,0,I,B789,298,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,07/12/2025 21:00,07/12/2025 21:15,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,08/12/2025 07:40,08/12/2025 07:10,REALIZADO,NaN,07/12/2025,Pontual,Antecipado
816,ACA,AIR CANADA,0085,0,I,B789,298,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,10/12/2025 21:00,10/12/2025 21:28,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...,11/12/2025 07:40,11/12/2025 07:50,REALIZADO,NaN,10/12/2025,Pontual,Pontual


In [563]:
vra_galeao_raw.info(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74954 entries, 0 to 74953
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   sg_empresa_icao        74954 non-null  object 
 1   nm_empresa             74954 non-null  object 
 2   nr_voo                 74954 non-null  object 
 3   cd_di                  74954 non-null  object 
 4   cd_tipo_linha          74954 non-null  object 
 5   sg_equipamento_icao    74954 non-null  object 
 6   nr_assentos_ofertados  74954 non-null  int64  
 7   sg_icao_origem         74954 non-null  object 
 8   nm_aerodromo_origem    74954 non-null  object 
 9   dt_partida_prevista    72439 non-null  object 
 10  dt_partida_real        73926 non-null  object 
 11  sg_icao_destino        74954 non-null  object 
 12  nm_aerodromo_destino   74954 non-null  object 
 13  dt_chegada_prevista    72439 non-null  object 
 14  dt_chegada_real        73926 non-null  object 
 15  ds

In [564]:
# Tipos de situação possível
vra_galeao_raw['ds_situacao_voo'].unique()

array(['REALIZADO', 'CANCELADO'], dtype=object)

In [565]:
# Conversão de datas para o formato apropriado
vra_galeao_silver_temp = vra_galeao_raw.copy()

vra_galeao_silver_temp['dt_partida_prevista_2'] = pd.to_datetime(vra_galeao_silver_temp['dt_partida_prevista'],dayfirst=True)
vra_galeao_silver_temp['dt_partida_real_2'] = pd.to_datetime(vra_galeao_silver_temp['dt_partida_real'], dayfirst=True)

# Extração do Ano e Mês. Considerando se o vôo foi realizado ou cancelado
vra_galeao_silver_temp['ANO'] = np.where(vra_galeao_silver_temp['ds_situacao_voo'] == 'REALIZADO', 
    vra_galeao_silver_temp['dt_partida_real_2'].dt.year, vra_galeao_silver_temp['dt_partida_prevista_2'].dt.year)

vra_galeao_silver_temp['MÊS'] = np.where(vra_galeao_silver_temp['ds_situacao_voo'] == 'REALIZADO', 
    vra_galeao_silver_temp['dt_partida_real_2'].dt.month, vra_galeao_silver_temp['dt_partida_prevista_2'].dt.month)

# Sumarização
vra_galeao_silver_temp = vra_galeao_silver_temp.groupby(by=['sg_empresa_icao', 'nm_empresa', 'ANO', 'MÊS', 'sg_icao_origem', 'sg_icao_destino', 
'ds_situacao_voo']).size().reset_index(name='VRA_DECOLAGEM')

# Pivotar a tabela sumarizada
vra_galeao_silver = vra_galeao_silver_temp.pivot_table(
    index=['sg_empresa_icao', 'nm_empresa', 'ANO', 'MÊS', 'sg_icao_origem', 'sg_icao_destino'], 
    columns='ds_situacao_voo', 
    values='VRA_DECOLAGEM',
    fill_value=0
).reset_index()

# Renomear campos
vra_galeao_silver = vra_galeao_silver.rename(columns={
    'CANCELADO': 'VRA_DECOLAGEM_CANCELADA', 'REALIZADO': 'VRA_DECOLAGEM_REALIZADO'})
vra_galeao_silver.head()

ds_situacao_voo,sg_empresa_icao,nm_empresa,ANO,MÊS,sg_icao_origem,sg_icao_destino,VRA_DECOLAGEM_CANCELADA,VRA_DECOLAGEM_REALIZADO
0,AAL,"AMERICAN AIRLINES, INC.",2025.0,7.0,KDFW,SBGL,0.0,1.0
1,AAL,"AMERICAN AIRLINES, INC.",2025.0,7.0,KMIA,SBGL,0.0,32.0
2,AAL,"AMERICAN AIRLINES, INC.",2025.0,7.0,SBGL,KMIA,1.0,30.0
3,AAL,"AMERICAN AIRLINES, INC.",2025.0,7.0,SBGL,SBEG,0.0,1.0
4,AAL,"AMERICAN AIRLINES, INC.",2025.0,7.0,SBGL,SBGR,0.0,1.0


### 4. Camada Ouro

Camada de dados refinada para análises e construção de indicadores.

In [568]:
# Seleção dos campos necessários para o analítico
colunas_gold = ['EMPRESA (SIGLA)', 'EMPRESA (NOME)', 'ANO', 'MÊS',
       'AEROPORTO DE ORIGEM (SIGLA)', 'AEROPORTO DE DESTINO (SIGLA)', 'AEROPORTO MERCADO (SIGLA)', 'NATUREZA',
       'ANAC_DECOLAGEM', 'SIROS_DECOLAGEM', 'VRA_DECOLAGEM_TOTAL', 'VRA_DECOLAGEM_CANCELADA', 'VRA_DECOLAGEM_REALIZADO', 
        'ANAC_PASSAGEIROS', 'ANAC_ASSENTOS', 'SIROS_ASSENTOS',]

# Left join Dados Estatísticos x VRA
galeao_gold_aux = pd.merge(
    anac_silver, 
    vra_galeao_silver, 
    left_on=['EMPRESA (SIGLA)', 'ANO', 'MÊS','AEROPORTO DE ORIGEM (SIGLA)', 'AEROPORTO DE DESTINO (SIGLA)'], 
    right_on=['sg_empresa_icao', 'ANO', 'MÊS', 'sg_icao_origem', 'sg_icao_destino'], 
    how='left'
)

# # Left join com Movimentações Siros
galeao_gold = pd.merge(
    galeao_gold_aux, 
    siros_mov_silver, 
    left_on=['EMPRESA (SIGLA)', 'ANO', 'MÊS','AEROPORTO DE ORIGEM (SIGLA)', 'AEROPORTO DE DESTINO (SIGLA)'], 
    right_on=['ICAO', 'Ano', 'Mês', 'Origem', 'Destino'], 
    how='left'
)

galeao_gold['VRA_DECOLAGEM_TOTAL'] =  galeao_gold['VRA_DECOLAGEM_CANCELADA'] +  galeao_gold['VRA_DECOLAGEM_REALIZADO'] 
galeao_gold = galeao_gold[colunas_gold].copy()
galeao_gold.head(5)

,EMPRESA (SIGLA),EMPRESA (NOME),ANO,MÊS,AEROPORTO DE ORIGEM (SIGLA),AEROPORTO DE DESTINO (SIGLA),AEROPORTO MERCADO (SIGLA),NATUREZA,ANAC_DECOLAGEM,SIROS_DECOLAGEM,VRA_DECOLAGEM_TOTAL,VRA_DECOLAGEM_CANCELADA,VRA_DECOLAGEM_REALIZADO,ANAC_PASSAGEIROS,ANAC_ASSENTOS,SIROS_ASSENTOS
0,AAL,"AMERICAN AIRLINES, INC.",2025,7,KDFW,SBGL,KDFW,INTERNACIONAL,1.0,NaN,1.0,0.0,1.0,0.0,285.0,NaN
1,AAL,"AMERICAN AIRLINES, INC.",2025,7,KMIA,SBGL,KMIA,INTERNACIONAL,31.0,31.0,32.0,0.0,32.0,8041.0,8464.0,8928.0
2,AAL,"AMERICAN AIRLINES, INC.",2025,7,SBGL,KMIA,KMIA,INTERNACIONAL,30.0,31.0,31.0,1.0,30.0,8279.0,8192.0,8928.0
3,AAL,"AMERICAN AIRLINES, INC.",2025,7,SBGL,SBEG,SBEG,INTERNACIONAL,1.0,NaN,1.0,0.0,1.0,0.0,272.0,NaN
4,AAL,"AMERICAN AIRLINES, INC.",2025,7,SBGL,SBGR,SBGR,INTERNACIONAL,1.0,NaN,1.0,0.0,1.0,0.0,285.0,NaN


In [570]:
# Exportação
galeao_gold.to_csv('galeao_gold.csv', index=False)